In [ ]:

from utils import get_device
from legacy.data_processing import get_loaders, class_cols
from VIS_V1 import create_model, train_model
import torch

import pandas as pd
from tqdm import tqdm

In [ ]:
device = get_device()

train_loader, val_loader, train_df, val_df = get_loaders(
    image_size=(384, 384),
    num_workers=0,
)
class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = len(class_cols_order)
model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")


class_counts = [int(train_df[c].sum()) for c in class_cols_order]

#base_model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")
#ckpt = torch.load("best_model.pth", map_location=device)
#base_model.load_state_dict(ckpt["model_state"])


model, history2 = train_model(
    train_loader,
    val_loader,
    num_classes=num_classes,
    device=device,
    epochs=15,                 
    class_counts=class_counts,
    #model=base_model,          
    save_path="best_model.pth" 
)


In [ ]:
# Evaluate on test set (if available)
# This cell will request loaders including the test set and compute test loss/accuracy
from legacy.data_processing import TASK_3_TEST_LABELS_DIR

# Try to get loaders and request the test CSV explicitly (module-level constant used as default)
loaders = get_loaders(
    image_size=(384, 384),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

# unpack safely depending on what get_loaders returned
test_loader = None
if isinstance(loaders, tuple) and len(loaders) == 6:
    train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
elif isinstance(loaders, tuple) and len(loaders) == 4:
    train_loader, val_loader, train_df, val_df = loaders
    test_loader = None
else:
    # unexpected shape
    test_loader = None

if test_loader is None:
    print("No test loader available. Provide a test CSV/images or adjust get_loaders to return a test loader.")
else:
    import torch
    import torch.nn as nn

    device = get_device()
    model.to(device)
    model.eval()

    criterion = nn.CrossEntropyLoss()
    total = 0
    correct = 0
    running_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            running_loss += loss.item() * xb.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)

    test_loss = running_loss / total if total > 0 else 0.0
    test_acc = correct / total if total > 0 else 0.0
    print(f"Test samples: {total}")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_acc:.4%}")



In [ ]:
for views, labels in train_loader:
    # views is a list/tuple of tensors: [weak_batch, strong_batch]
    weak_imgs, strong_imgs = views[0].to(device), views[1].to(device)
    labels = labels.to(device)

# Case 2 with data augmentation

In [ ]:
from legacy.data_process_V3 import get_loaders, class_cols
from legacy.data_processing import TASK_3_TEST_LABELS_DIR
from VIS_V1 import create_model, train_model
from utils import get_device

device = get_device()

loaders = get_loaders(
    image_size=(384, 384),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

# unpack safely depending on what get_loaders returned
test_loader = None
if isinstance(loaders, tuple) and len(loaders) == 6:
    train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
elif isinstance(loaders, tuple) and len(loaders) == 4:
    train_loader, val_loader, train_df, val_df = loaders
    test_loader = None
else:
    # unexpected shape
    test_loader = None
class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = len(class_cols_order)
model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")


class_counts = [int(train_df[c].sum()) for c in class_cols_order]

#base_model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")
#ckpt = torch.load("best_model.pth", map_location=device)
#base_model.load_state_dict(ckpt["model_state"])


model, history2 = train_model(
    train_loader,
    val_loader,
    num_classes=num_classes,
    device=device,
    epochs=20,                 
    class_counts=class_counts,
    #model=base_model,          
    save_path="model_v1_fullaug.pth" 
)


Epoch 1/20 train_loss=0.1900 val_loss=3.1116 train_acc=0.0570 val_acc=0.0311
  ✓ New best val_acc=0.0311, model saved to model_v1_fullaug.pth


Epoch 2/20 train_loss=0.1311 val_loss=2.4236 train_acc=0.0813 val_acc=0.1554
  ✓ New best val_acc=0.1554, model saved to model_v1_fullaug.pth


Epoch 3/20 train_loss=0.1197 val_loss=2.5850 train_acc=0.1050 val_acc=0.1710
  ✓ New best val_acc=0.1710, model saved to model_v1_fullaug.pth


Epoch 4/20 train_loss=0.1160 val_loss=1.8506 train_acc=0.1095 val_acc=0.3782
  ✓ New best val_acc=0.3782, model saved to model_v1_fullaug.pth


Epoch 5/20 train_loss=0.1120 val_loss=3.5259 train_acc=0.1311 val_acc=0.0829


...

Epoch 16/20 train_loss=0.0555 val_loss=1.8375 train_acc=0.2993 val_acc=0.3886


Epoch 17/20 train_loss=0.0510 val_loss=1.7374 train_acc=0.3199 val_acc=0.4093


Epoch 18/20 train_loss=0.0460 val_loss=1.8662 train_acc=0.3320 val_acc=0.3264


Epoch 19/20 train_loss=0.0419 val_loss=1.8296 train_acc=0.3447 val_acc=0.3627


Epoch 20/20 train_loss=0.0409 val_loss=1.8200 train_acc=0.3413 val_acc=0.3420


In [8]:
try:
    from sklearn.metrics import (
        balanced_accuracy_score,
        accuracy_score,
        precision_recall_fscore_support,
        confusion_matrix,
        classification_report,
    )
    _has_sklearn = True
except Exception:
    _has_sklearn = False

try:
    from tqdm import tqdm
except Exception:
    # if tqdm isn't available, just use a dummy wrapper
    def tqdm(x):
        return x

if 'test_loader' not in globals() or test_loader is None:
    print("No test loader available. Skipping metrics computation.")
else:
    device = get_device()
    model.to(device)
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            out = model(xb)
            preds = out.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_targets.append(yb.cpu())
    if len(all_preds) == 0:
        print("No test samples found.")
    else:
        import numpy as np
        y_pred = np.concatenate([a.numpy() for a in all_preds]) if hasattr(all_preds[0], 'numpy') else np.concatenate(all_preds)
        y_true = np.concatenate([a.numpy() for a in all_targets]) if hasattr(all_targets[0], 'numpy') else np.concatenate(all_targets)

        # -------- sklearn path --------
        if _has_sklearn:
            try:
                bal_acc = balanced_accuracy_score(y_true, y_pred)
                acc = accuracy_score(y_true, y_pred)

                # macro-averaged precision/recall/F1
                prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
                    y_true, y_pred, average="macro", zero_division=0
                )

                cm = confusion_matrix(y_true, y_pred)

                print(f"Accuracy:          {acc:.4%}")
                print(f"Balanced accuracy: {bal_acc:.4%}")
                print(f"Macro precision:   {prec_macro:.4%}")
                print(f"Macro recall:      {rec_macro:.4%}")
                print(f"Macro F1:          {f1_macro:.4%}")
                print("Confusion matrix:\n", cm)

                # Optional: per-class breakdown
                # print(classification_report(y_true, y_pred))

            except Exception as e:
                print("sklearn failed to compute metrics, falling back. Error:", e)
                _has_sklearn = False

        # -------- fallback (no sklearn) --------
        if not _has_sklearn:
            # accuracy
            acc = float((y_true == y_pred).mean()) if len(y_true) > 0 else float("nan")

            # confusion matrix
            if len(y_true) > 0:
                num_classes = int(max(y_true.max(), y_pred.max())) + 1
            else:
                num_classes = 0

            cm = np.zeros((num_classes, num_classes), dtype=int)
            for t, p in zip(y_true, y_pred):
                cm[int(t), int(p)] += 1

            # per-class precision / recall / F1
            recalls = []
            precisions = []


Accuracy:          27.3148%
Balanced accuracy: 46.8622%
Macro precision:   35.0621%
Macro recall:      46.8622%
Macro F1:          30.1611%
Confusion matrix:
 [[ 89   0  10  45  11  14   2]
 [262 167  35 121 191 124   9]
 [  8   0  35  32   3  12   3]
 [  2   0   1  33   1   6   0]
 [ 62   0  21  60  40  30   4]
 [  1   0   6   8   2  26   1]
 [  0   1   4   0   1   6  23]]


# convnextv2

Model is way to massive to train it locally efficiently...

In [ ]:
from legacy.data_process_V3 import get_loaders, class_cols
from legacy.convnextv2 import create_model, train_model
from legacy.data_processing import TASK_3_TEST_LABELS_DIR
from utils import get_device


device = get_device()

loaders = get_loaders(
    image_size=(384, 384),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

# unpack safely depending on what get_loaders returned
test_loader = None
if isinstance(loaders, tuple) and len(loaders) == 6:
    train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
elif isinstance(loaders, tuple) and len(loaders) == 4:
    train_loader, val_loader, train_df, val_df = loaders
    test_loader = None
else:
    # unexpected shape
    test_loader = None

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = len(class_cols_order)

model = create_model(
    num_classes=num_classes,
    model_name="convnextv2_base",  
)

# class counts for BalancedFocalLoss
class_counts = [int(train_df[c].sum()) for c in class_cols_order]

# Base model to (optionally) resume from checkpoint
#base_model = create_model(
#    num_classes=num_classes,
#model_name="convnextv2_base",  # <- same ConvNeXtV2 backbone
#)

#ckpt = torch.load("best_model_convnextv2.pth", map_location=device)
#base_model.load_state_dict(ckpt["model_state"])

# Fine-tune / continue training ConvNeXtV2
model, history2 = train_model(
    train_loader,
    val_loader,
    num_classes=num_classes,
    device=device,
    epochs=20,
    class_counts=class_counts,
    #model=base_model,          
    save_path="best_model_convnextv2.pth"
      )



# VIS V2 

In [ ]:
import os
import torch

from legacy.data_process_augmented import get_loaders, class_cols
from vis_V2 import create_vis_model, train_model
from legacy.data_processing import TASK_3_TEST_LABELS_DIR
from utils import get_device


device = get_device()
print(f"Using device: {device}")
loaders = get_loaders(
    image_size=(600, 450),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)
# unpack safely depending on what get_loaders returned
train_loader = None
val_loader = None
test_loader = None
train_df = None
val_df = None
test_df = None
if isinstance(loaders, tuple) and len(loaders) == 6:
    train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
elif isinstance(loaders, tuple) and len(loaders) == 4:
    train_loader, val_loader, train_df, val_df = loaders
    test_loader = None
else:
    raise RuntimeError(
        f"Unexpected loaders return shape: type={type(loaders)}, value={loaders}"
    )
# class setup
class_cols_order = class_cols  # keep source of truth from data_process_V3
num_classes = len(class_cols_order)
if train_df is None:
    raise RuntimeError("train_df is None; get_loaders did not return a train_df.")
class_counts = [int(train_df[c].sum()) for c in class_cols_order]
print("Class counts:", class_counts)
model_name = "vit_base_patch16_384"
base_model = create_vis_model(
    num_classes=num_classes,
    model_name=model_name,
    pretrained=True,
    in_channels=3,
)
base_model.to(device)
ckpt_path = "vis_model_v2.pth"
if os.path.exists(ckpt_path):
    print(f"Loading checkpoint from {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    # handle both {"model_state": ...} and raw state_dict checkpoints
    if isinstance(ckpt, dict) and "model_state" in ckpt:
        base_model.load_state_dict(ckpt["model_state"])
    else:
        base_model.load_state_dict(ckpt)
else:
    print(f"No checkpoint found at {ckpt_path}, training from scratch.")
model, history = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=num_classes,
    model_name=model_name,
    device=device,
    epochs=10,
    class_counts=class_counts,  # BalancedFocalLoss inside if provided
    model=base_model,           # start from VIS model (possibly ckpt-loaded)
    save_path=ckpt_path,        # save back to same checkpoint file
)
print("Training finished.")
print("History keys:", history.keys())


Using CSV_PATH: data/Task_3/ISIC2018_Task3_Training_GroundTruth.csv
Using IMAGES_DIR: data/Task_3/Train_images
Using VAL CSV: data/Task_3/ISIC2018_Task3_Validation_GroundTruth.csv
Using VAL IMAGES_DIR: data/Task_3/Validation_images
Using TEST CSV: data/Task_3/ISIC2018_Task3_Test_GroundTruth.csv
Using TEST IMAGES_DIR: data/Task_3/Test_images
Using CSV_PATH: data/Task_3/ISIC2018_Task3_Training_GroundTruth.csv
Using IMAGES_DIR: data/Task_3/Train_images
Using VAL CSV: data/Task_3/ISIC2018_Task3_Validation_GroundTruth.csv
Using VAL IMAGES_DIR: data/Task_3/Validation_images
Using TEST CSV: data/Task_3/ISIC2018_Task3_Test_GroundTruth.csv
Using TEST IMAGES_DIR: data/Task_3/Test_images
Using device: mps
Class counts: [1113, 6705, 514, 327, 1099, 115, 142]
No checkpoint found at vis_model_v2.pth, training from scratch.


KeyboardInterrupt: 